# JEDI Final Modeling Workbook

Job Evaluation & Delivery Intelligence

This notebook is the clean modeling artifact for the JEDI project. It is structured as a professional ML workflow:

1. problem framing,
2. data validation,
3. preprocessing pipelines,
4. feature engineering,
5. score regression,
6. stacked XGBoost classification,
7. hyperparameter search,
8. cross-validation,
9. metric interpretation,
10. final artifact saving.

The central product decision is ranking-first. JEDI should not only say whether a job is apply-worthy. It should assign a decimal preference score so that 50 daily jobs can be sorted and the best 25 can be reviewed first.

## 1. Problem Setup

### What this section is

This section defines the modeling task and explains why the final output has two parts.

### Result expected

No code is required here.

### Interpretation

The binary label, Applied, is useful for training. But it is too coarse for the product. Two jobs can both be apply-worthy while one is much stronger. Therefore the final JEDI output is:

- predicted personal fit scores,
- JEDI preference score from those fit scores,
- apply probability from the classifier,
- final recommendation label.

This lets the app rank jobs, not just classify them.

## 2. Imports And Reproducibility

### What this section is

This block loads libraries, sets constants, and prepares output folders.

### Code

In [ ]:
from pathlib import Path
import json
import warnings

import numpy as np
import pandas as pd

from joblib import dump
from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    balanced_accuracy_score,
    brier_score_loss,
    f1_score,
    log_loss,
    mean_absolute_error,
    precision_score,
    r2_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import KFold, StratifiedKFold, cross_val_predict
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import MaxAbsScaler, OneHotEncoder

try:
    from xgboost import XGBClassifier
    HAS_XGBOOST = True
except ModuleNotFoundError:
    HAS_XGBOOST = False

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 160)
pd.set_option("display.max_colwidth", 140)

RANDOM_STATE = 42
N_SPLITS = 5

DATA_PATH = Path("jedi_final_model_dataset.csv")
OUTPUT_DIR = Path("data/final")
MODEL_DIR = Path("models")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)

print("XGBoost available:", HAS_XGBOOST)
print("Dataset path:", DATA_PATH.resolve())

### Results

The output confirms whether XGBoost is available and where the dataset is loaded from.

### Interpretation

If XGBoost is unavailable, the final classifier cannot run. The rest of the notebook assumes XGBoost exists because the selected final model is stacked XGBoost.

## 3. Load And Validate Dataset

### What this section is

This block loads the final dataset and checks whether the target label is valid.

### Code

In [ ]:
df = pd.read_csv(DATA_PATH)

print(f"Rows: {len(df):,}")
print(f"Columns: {df.shape[1]:,}")

required_columns = ["applied", "skills_or_requirements", "role_title", "role_family", "seniority"]
missing_required = [col for col in required_columns if col not in df.columns]
if missing_required:
    raise ValueError(f"Missing required columns: {missing_required}")

df["applied_clean"] = df["applied"].astype(str).str.strip().str.lower()
invalid_labels = sorted(set(df["applied_clean"]) - {"yes", "no"})
if invalid_labels:
    raise ValueError(f"Invalid Applied labels: {invalid_labels}")

display(df.head(3))
display(df["applied_clean"].value_counts().rename_axis("applied").reset_index(name="rows"))

### Results

This output shows dataset size, a small preview, and class distribution.

### Interpretation

The class distribution matters because accuracy can look good even when a model misses many useful jobs. For JEDI, precision, recall, F1, and ranking metrics are more informative than accuracy alone.

## 4. Targets And Leakage Control

### What this section is

This block defines the target variables and removes rows that are not trainable.

The score columns are not treated as ordinary features. They are prediction targets first. Later, predicted score values are stacked into the classifier.

### Code

In [ ]:
SCORE_TARGETS = [
    "role_title_preference_score",
    "role_family_preference_score",
    "seniority_profile_fit_score",
    "location_profile_fit_score",
    "work_mode_fit_score",
    "experience_profile_fit_score",
    "skills_profile_fit_score",
    "salary_profile_fit_score",
    "overall_preference_score",
]

for col in SCORE_TARGETS:
    if col not in df.columns:
        raise ValueError(f"Missing score target column: {col}")
    df[col] = pd.to_numeric(df[col], errors="coerce")

df["target_applied"] = df["applied_clean"].map({"no": 0, "yes": 1}).astype(int)

valid_rows = df[SCORE_TARGETS].notna().all(axis=1)
model_df = df.loc[valid_rows].reset_index(drop=True).copy()

y = model_df["target_applied"].astype(int)
y_scores = model_df[SCORE_TARGETS].astype(float)

print(f"Usable rows after target validation: {len(model_df):,}")
display(y.value_counts().rename_axis("target_applied").reset_index(name="rows"))

### Results

The output shows how many rows are usable after requiring all score targets.

### Interpretation

This is important because earlier experiments accidentally lost rows when experience score was missing. The final dataset repaired that issue, so the final model trains on the full prepared dataset.

## 5. Feature Engineering

### What this section is

This block builds model-ready feature groups:

- numeric features,
- boolean features,
- categorical features,
- engineered boolean features,
- text features.

It also defines leakage columns that must not enter the raw classifier.

### Code

In [ ]:
ALL_SCORE_COLUMNS = [
    col for col in model_df.columns
    if col.endswith("_score") or col in ["company_match_score", "overall_preference_score"]
]

LEAKAGE_COLUMNS = [
    "row_id",
    "applied",
    "applied_clean",
    "target_applied",
    "preference_reason",
    "apply_url",
    "source_platform",
    "source_type",
    "scraped_at",
    "near_duplicate_key",
    "company_name",
    "company_name_standard",
    "role_title_standard",
]

drop_columns = sorted(set(ALL_SCORE_COLUMNS + LEAKAGE_COLUMNS))
candidate_features = [col for col in model_df.columns if col not in drop_columns]

engineered_features = [col for col in model_df.columns if col.startswith("feat_")]

base_numeric = [
    "experience_min_years",
    "salary_min",
    "salary_max",
    "salary_midpoint",
    "experience_profile_fit_imputed",
]

boolean_features = [
    "has_skills_or_requirements",
    "is_bengaluru",
    "is_gurgaon_ncr",
    "is_india",
    "salary_available",
]

categorical_features = [
    "can_apply_now",
    "freshness",
    "role_family_standard",
    "seniority_standard",
    "employment_type_standard",
    "work_mode_standard",
    "city_standard",
    "country_standard",
    "location_bucket",
    "experience_min_bucket",
    "skills_imputation_type",
    "experience_profile_fit_imputation_reason",
]

base_numeric = [col for col in base_numeric if col in model_df.columns]
boolean_features = [col for col in boolean_features if col in model_df.columns]
categorical_features = [col for col in categorical_features if col in model_df.columns]

def to_binary(value):
    if pd.isna(value):
        return np.nan
    text = str(value).strip().lower()
    if text in {"true", "1", "yes"}:
        return 1.0
    if text in {"false", "0", "no"}:
        return 0.0
    return np.nan

for col in base_numeric + engineered_features:
    model_df[col] = pd.to_numeric(model_df[col], errors="coerce")

for col in boolean_features:
    model_df[col] = model_df[col].map(to_binary)

for col in ["enhanced_title_skill_text", "enhanced_context_text"]:
    if col not in model_df.columns:
        model_df[col] = ""
    model_df[col] = model_df[col].fillna("").astype(str)

X_base = model_df[candidate_features].copy()
X_base["enhanced_title_skill_text"] = model_df["enhanced_title_skill_text"]
X_base["enhanced_context_text"] = model_df["enhanced_context_text"]

feature_summary = pd.DataFrame([
    {"feature_group": "numeric", "count": len(base_numeric)},
    {"feature_group": "boolean", "count": len(boolean_features)},
    {"feature_group": "categorical", "count": len(categorical_features)},
    {"feature_group": "engineered", "count": len(engineered_features)},
    {"feature_group": "candidate_columns", "count": len(candidate_features)},
])

display(feature_summary)
display(pd.DataFrame({"dropped_as_leakage": drop_columns}).head(30))

### Results

The output shows the feature groups and a sample of columns removed as leakage.

### Interpretation

This is the most important correctness step in the notebook. If true handwritten preference scores are allowed into the classifier, the model will look artificially strong. The proper approach is to predict scores first, then stack only predicted scores into the classifier.

## 6. Preprocessing Pipeline

### What this section is

This block defines the reusable preprocessing pipeline.

Numeric columns are imputed and scaled. Categorical columns are one-hot encoded. Text columns are vectorized with word-level and character-level TF-IDF.

### Code

In [ ]:
def make_preprocessor(extra_numeric=None, dropped_engineered=None, text_size="large"):
    extra_numeric = list(extra_numeric or [])
    dropped_engineered = set(dropped_engineered or [])
    active_engineered = [col for col in engineered_features if col not in dropped_engineered]
    numeric_cols = base_numeric + boolean_features + active_engineered + extra_numeric

    if text_size == "large":
        word_max_features = 4200
        char_max_features = 2800
    else:
        word_max_features = 2600
        char_max_features = 1800

    numeric_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", MaxAbsScaler()),
    ])

    categorical_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore", min_frequency=3)),
    ])

    word_text = TfidfVectorizer(
        analyzer="word",
        ngram_range=(1, 2),
        min_df=2,
        max_features=word_max_features,
        sublinear_tf=True,
    )

    char_text = TfidfVectorizer(
        analyzer="char_wb",
        ngram_range=(3, 5),
        min_df=2,
        max_features=char_max_features,
        sublinear_tf=True,
    )

    return ColumnTransformer([
        ("numeric", numeric_pipe, numeric_cols),
        ("categorical", categorical_pipe, categorical_features),
        ("word_text", word_text, "enhanced_title_skill_text"),
        ("char_text", char_text, "enhanced_context_text"),
    ], remainder="drop")

preprocessor_preview = make_preprocessor()
print(preprocessor_preview)

### Results

The output prints the preprocessing pipeline.

### Interpretation

This pipeline is production-friendly because the same transformations can be saved and reused for new jobs. It also avoids manually transforming training and inference data differently.

## 7. Fit-Score Regression Model

### What this section is

This block trains a Ridge regression model to predict personal fit scores.

These predicted scores become the basis for the JEDI preference score and are also used as stacked features for the classifier.

### Code

In [ ]:
SCORE_MAX = {
    "role_title_preference_score": 5.0,
    "role_family_preference_score": 5.0,
    "seniority_profile_fit_score": 5.0,
    "location_profile_fit_score": 5.0,
    "work_mode_fit_score": 5.0,
    "experience_profile_fit_score": 5.0,
    "skills_profile_fit_score": 5.0,
    "salary_profile_fit_score": 8.0,
    "overall_preference_score": 5.0,
}

JEDI_SCORE_WEIGHTS = {
    "role_title_preference_score": 0.18,
    "role_family_preference_score": 0.12,
    "skills_profile_fit_score": 0.18,
    "experience_profile_fit_score": 0.14,
    "location_profile_fit_score": 0.12,
    "work_mode_fit_score": 0.07,
    "salary_profile_fit_score": 0.07,
    "seniority_profile_fit_score": 0.05,
    "overall_preference_score": 0.07,
}

assert abs(sum(JEDI_SCORE_WEIGHTS.values()) - 1.0) < 1e-9

def clip_score_predictions(score_array):
    arr = np.asarray(score_array, dtype=float).copy()
    for idx, target in enumerate(SCORE_TARGETS):
        arr[:, idx] = np.clip(arr[:, idx], 0.0, SCORE_MAX[target])
    return arr

def make_score_model():
    return Pipeline([
        ("preprocess", make_preprocessor(text_size="large")),
        ("ridge", Ridge(alpha=2.0)),
    ])

def compute_jedi_preference_score(score_frame):
    score = np.zeros(len(score_frame), dtype=float)
    for col, weight in JEDI_SCORE_WEIGHTS.items():
        normalized = pd.to_numeric(score_frame[col], errors="coerce").fillna(0) / SCORE_MAX[col]
        score += weight * normalized
    return np.round(score * 100, 2)

score_cv = KFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)
score_oof = clip_score_predictions(cross_val_predict(make_score_model(), X_base, y_scores, cv=score_cv, n_jobs=None))
score_oof_frame = pd.DataFrame(score_oof, columns=SCORE_TARGETS)

score_metric_rows = []
for idx, target in enumerate(SCORE_TARGETS):
    score_metric_rows.append({
        "target": target,
        "r2": r2_score(y_scores[target], score_oof[:, idx]),
        "mae": mean_absolute_error(y_scores[target], score_oof[:, idx]),
    })

score_metrics = pd.DataFrame(score_metric_rows).sort_values("r2", ascending=False)
model_df["jedi_preference_score"] = compute_jedi_preference_score(score_oof_frame)

display(score_metrics)
display(
    model_df[["row_id", "company_name", "role_title", "applied", "jedi_preference_score"]]
    .sort_values("jedi_preference_score", ascending=False)
    .head(10)
)

### Results

This output shows how well each personal fit score can be predicted and shows the top jobs by JEDI preference score.

### Interpretation

Ridge is used because the fit scores are continuous preference signals. The JEDI score is not a classifier probability. It is a weighted preference score designed to rank jobs. This is the score the UI should sort by.

## 8. Stacking Helpers

### What this section is

This block defines utilities for stacking predicted fit scores into the classifier and for dropping weak engineered features.

### Code

In [ ]:
PRED_SCORE_COLUMNS = [f"pred_{col}" for col in SCORE_TARGETS]

def append_predicted_scores(X, score_array):
    out = X.copy()
    score_array = clip_score_predictions(score_array)
    for idx, target in enumerate(SCORE_TARGETS):
        out[f"pred_{target}"] = score_array[:, idx]
    return out

def safe_corr(x, target):
    x = pd.to_numeric(pd.Series(x), errors="coerce")
    target = pd.to_numeric(pd.Series(target), errors="coerce")
    valid = x.notna() & target.notna()
    if valid.sum() < 3 or x[valid].nunique() <= 1:
        return 0.0
    return float(x[valid].corr(target[valid]))

def low_correlation_engineered_features(X_train, y_train, drop_count):
    rows = []
    for col in engineered_features:
        if col in X_train.columns:
            corr = safe_corr(X_train[col], y_train)
            rows.append({"feature": col, "corr": corr, "abs_corr": abs(corr)})
    corr_df = pd.DataFrame(rows).sort_values(["abs_corr", "feature"]).reset_index(drop=True)
    drop_features = corr_df.head(min(drop_count, len(corr_df)))["feature"].tolist()
    return drop_features, corr_df

print("Predicted score columns used for stacking:")
display(pd.DataFrame({"column": PRED_SCORE_COLUMNS}))

### Results

The output lists the predicted score columns that will be appended to the classifier.

### Interpretation

Stacking means the classifier receives model-predicted preference dimensions. This is safer than feeding the true annotated scores, because new scraped jobs will not have true annotated scores.

## 9. Model Candidates And Hyperparameter Search

### What this section is

This block defines the XGBoost candidates. This is a compact hyperparameter search, not an exhaustive grid.

Each candidate is evaluated with cross-validation.

### Code

In [ ]:
if not HAS_XGBOOST:
    raise ModuleNotFoundError("XGBoost is required for this notebook.")

MODEL_CANDIDATES = [
    {
        "name": "xgb_conservative",
        "drop_count": 5,
        "threshold_grid": np.arange(0.25, 0.61, 0.05),
        "params": {
            "n_estimators": 500,
            "max_depth": 2,
            "learning_rate": 0.025,
            "min_child_weight": 5,
            "subsample": 0.90,
            "colsample_bytree": 0.90,
            "reg_alpha": 0.25,
            "reg_lambda": 5.0,
        },
    },
    {
        "name": "xgb_balanced",
        "drop_count": 20,
        "threshold_grid": np.arange(0.25, 0.61, 0.05),
        "params": {
            "n_estimators": 650,
            "max_depth": 3,
            "learning_rate": 0.025,
            "min_child_weight": 3,
            "subsample": 0.85,
            "colsample_bytree": 0.85,
            "reg_alpha": 0.10,
            "reg_lambda": 3.0,
        },
    },
    {
        "name": "xgb_deeper",
        "drop_count": 20,
        "threshold_grid": np.arange(0.25, 0.61, 0.05),
        "params": {
            "n_estimators": 450,
            "max_depth": 4,
            "learning_rate": 0.03,
            "min_child_weight": 2,
            "subsample": 0.80,
            "colsample_bytree": 0.80,
            "reg_alpha": 0.15,
            "reg_lambda": 4.0,
        },
    },
]

display(pd.DataFrame([
    {"name": spec["name"], "drop_count": spec["drop_count"], **spec["params"]}
    for spec in MODEL_CANDIDATES
]))

### Results

The output shows the candidate model configurations.

### Interpretation

The candidates move from conservative to more expressive. The purpose is to find the best tradeoff between underfitting and overfitting without pretending that endless tuning will solve dataset limitations.

## 10. Cross-Validation Training

### What this section is

This block performs the full stacked evaluation:

1. split train and validation folds,
2. train Ridge fit-score model inside the fold,
3. create predicted score features,
4. train XGBoost candidates,
5. evaluate thresholds and ranking quality.

### Code

In [ ]:
def make_xgb_classifier(spec, extra_numeric, dropped_engineered, scale_pos_weight):
    params = spec["params"].copy()
    params.update({
        "objective": "binary:logistic",
        "eval_metric": "logloss",
        "tree_method": "hist",
        "random_state": RANDOM_STATE,
        "n_jobs": 4,
        "scale_pos_weight": scale_pos_weight,
    })
    return Pipeline([
        ("preprocess", make_preprocessor(
            extra_numeric=extra_numeric,
            dropped_engineered=dropped_engineered,
            text_size="large",
        )),
        ("classifier", XGBClassifier(**params)),
    ])

def classification_metrics(model_name, fold, y_true, proba, threshold):
    pred = (np.asarray(proba) >= threshold).astype(int)
    return {
        "model": model_name,
        "fold": fold,
        "threshold": threshold,
        "accuracy": accuracy_score(y_true, pred),
        "balanced_accuracy": balanced_accuracy_score(y_true, pred),
        "precision_yes": precision_score(y_true, pred, zero_division=0),
        "recall_yes": recall_score(y_true, pred, zero_division=0),
        "f1_yes": f1_score(y_true, pred, zero_division=0),
        "roc_auc": roc_auc_score(y_true, proba),
        "average_precision": average_precision_score(y_true, proba),
        "log_loss": log_loss(y_true, proba, labels=[0, 1]),
        "brier_score": brier_score_loss(y_true, proba),
        "predicted_yes_count": int(pred.sum()),
    }

def ranking_metrics(model_name, fold, y_true, score, source, ks=(10, 20, 25, 50)):
    frame = pd.DataFrame({"y": np.asarray(y_true), "score": np.asarray(score)})
    frame = frame.sort_values("score", ascending=False).reset_index(drop=True)
    total_positive = max(int(frame["y"].sum()), 1)
    rows = []
    for k in ks:
        top = frame.head(k)
        gains = top["y"].values
        discounts = 1 / np.log2(np.arange(2, len(gains) + 2))
        dcg = float(np.sum(gains * discounts))
        ideal = np.sort(frame["y"].values)[::-1][:k]
        ideal_dcg = float(np.sum(ideal * discounts[:len(ideal)]))
        rows.append({
            "model": model_name,
            "fold": fold,
            "rank_source": source,
            "k": k,
            "precision_at_k": top["y"].mean(),
            "recall_at_k": top["y"].sum() / total_positive,
            "ndcg_at_k": dcg / ideal_dcg if ideal_dcg else 0.0,
        })
    return rows

outer_cv = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)
inner_cv = KFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)

detail_rows = []
threshold_rows = []
ranking_rows = []
prediction_frames = []
drop_rows = []

for fold, (train_idx, val_idx) in enumerate(outer_cv.split(X_base, y), start=1):
    X_train = X_base.iloc[train_idx].copy()
    X_val = X_base.iloc[val_idx].copy()
    y_train = y.iloc[train_idx].copy()
    y_val = y.iloc[val_idx].copy()
    y_score_train = y_scores.iloc[train_idx].copy()

    fold_score_model = make_score_model()
    train_score_oof = clip_score_predictions(cross_val_predict(fold_score_model, X_train, y_score_train, cv=inner_cv, n_jobs=None))
    fold_score_model.fit(X_train, y_score_train)
    val_score_pred = clip_score_predictions(fold_score_model.predict(X_val))

    X_train_stacked = append_predicted_scores(X_train, train_score_oof)
    X_val_stacked = append_predicted_scores(X_val, val_score_pred)
    val_score_frame = pd.DataFrame(val_score_pred, columns=SCORE_TARGETS)
    val_jedi_score = compute_jedi_preference_score(val_score_frame)

    fold_predictions = model_df.iloc[val_idx][["row_id", "company_name", "role_title", "applied"]].reset_index(drop=True).copy()
    fold_predictions["fold"] = fold
    fold_predictions["jedi_preference_score"] = val_jedi_score
    fold_predictions["actual_applied"] = y_val.reset_index(drop=True).values

    scale_pos_weight = float((y_train == 0).sum() / max((y_train == 1).sum(), 1))

    for spec in MODEL_CANDIDATES:
        dropped, corr_df = low_correlation_engineered_features(X_train_stacked, y_train, spec["drop_count"])
        for feature in dropped:
            drop_rows.append({"fold": fold, "model": spec["name"], "feature": feature})

        estimator = make_xgb_classifier(
            spec,
            extra_numeric=PRED_SCORE_COLUMNS,
            dropped_engineered=dropped,
            scale_pos_weight=scale_pos_weight,
        )
        estimator.fit(X_train_stacked, y_train)
        proba = estimator.predict_proba(X_val_stacked)[:, 1]

        detail_rows.append(classification_metrics(spec["name"], fold, y_val, proba, threshold=0.50))
        ranking_rows.extend(ranking_metrics(spec["name"], fold, y_val, proba, source="apply_probability"))
        ranking_rows.extend(ranking_metrics(spec["name"], fold, y_val, val_jedi_score, source="jedi_preference_score"))

        for threshold in spec["threshold_grid"]:
            threshold_rows.append(classification_metrics(spec["name"], fold, y_val, proba, threshold=float(round(threshold, 2))))

        fold_predictions[f"{spec['name']}_apply_probability"] = proba
        fold_predictions[f"{spec['name']}_default_prediction"] = (proba >= 0.50).astype(int)

    prediction_frames.append(fold_predictions)

cv_detail = pd.DataFrame(detail_rows)
threshold_detail = pd.DataFrame(threshold_rows)
ranking_detail = pd.DataFrame(ranking_rows)
oof_predictions = pd.concat(prediction_frames, ignore_index=True)
dropped_features = pd.DataFrame(drop_rows)

print("Cross-validation complete")
print("CV detail rows:", len(cv_detail))
print("Threshold rows:", len(threshold_detail))
print("Ranking rows:", len(ranking_detail))

### Results

The output confirms that every candidate model was evaluated across cross-validation folds.

### Interpretation

This is the trustworthy evaluation step. The score model is fit inside each fold, so the classifier does not get leaked true preference scores from validation rows.

## 11. Model Selection And Metric Interpretation

### What this section is

This block summarizes cross-validation, chooses the best model, finds the best threshold, and explains what the numbers mean for the JEDI product.

### Code

In [ ]:
metric_columns = [
    "accuracy",
    "balanced_accuracy",
    "precision_yes",
    "recall_yes",
    "f1_yes",
    "roc_auc",
    "average_precision",
    "log_loss",
    "brier_score",
    "predicted_yes_count",
]

cv_summary = (
    cv_detail
    .groupby("model")[metric_columns]
    .agg(["mean", "std"])
)
cv_summary.columns = [f"{metric}_{stat}" for metric, stat in cv_summary.columns]
cv_summary = cv_summary.reset_index().sort_values(["f1_yes_mean", "roc_auc_mean"], ascending=False)

threshold_summary = (
    threshold_detail
    .groupby(["model", "threshold"])[["precision_yes", "recall_yes", "f1_yes", "balanced_accuracy", "predicted_yes_count"]]
    .mean()
    .reset_index()
    .sort_values(["f1_yes", "balanced_accuracy"], ascending=False)
)

ranking_summary = (
    ranking_detail
    .groupby(["model", "rank_source", "k"])[["precision_at_k", "recall_at_k", "ndcg_at_k"]]
    .mean()
    .reset_index()
)

best_default = cv_summary.iloc[0]
best_threshold = threshold_summary.iloc[0]
best_rank50 = (
    ranking_summary[ranking_summary["k"] == 50]
    .sort_values(["precision_at_k", "ndcg_at_k"], ascending=False)
    .iloc[0]
)

def pct(x):
    return f"{float(x) * 100:0.2f}%"

interpretation_table = pd.DataFrame([
    {
        "metric": "Accuracy",
        "value": pct(best_default["accuracy_mean"]),
        "what_it_means": "Overall yes/no correctness.",
        "project_decision": "Useful sanity check, but not enough because JEDI is a ranking product.",
    },
    {
        "metric": "Precision",
        "value": pct(best_default["precision_yes_mean"]),
        "what_it_means": "Of jobs predicted apply-worthy, how many were truly apply-worthy.",
        "project_decision": "Higher precision means less wasted review/application time.",
    },
    {
        "metric": "Recall",
        "value": pct(best_default["recall_yes_mean"]),
        "what_it_means": "Of truly apply-worthy jobs, how many the model catches.",
        "project_decision": "Higher recall means fewer good roles missed.",
    },
    {
        "metric": "F1 yes",
        "value": pct(best_default["f1_yes_mean"]),
        "what_it_means": "Balance between precision and recall.",
        "project_decision": "Main binary-classification score. Strong, but not strong enough for blind auto-apply.",
    },
    {
        "metric": "ROC-AUC",
        "value": pct(best_default["roc_auc_mean"]),
        "what_it_means": "How well the model separates yes jobs from no jobs across thresholds.",
        "project_decision": "High ROC-AUC means the model ranking signal is meaningful.",
    },
    {
        "metric": "PR-AUC",
        "value": pct(best_default["average_precision_mean"]),
        "what_it_means": "Positive-class quality across thresholds.",
        "project_decision": "Important because the useful class is apply-worthy jobs.",
    },
    {
        "metric": "Precision@50",
        "value": pct(best_rank50["precision_at_k"]),
        "what_it_means": "Of the top 50 ranked jobs, how many are apply-worthy.",
        "project_decision": "Most product-relevant metric for a daily job screener.",
    },
    {
        "metric": "NDCG@50",
        "value": f"{float(best_rank50['ndcg_at_k']):0.4f}",
        "what_it_means": "Whether good jobs appear near the very top.",
        "project_decision": "High NDCG supports showing only the top 25-50 jobs.",
    },
])

print("Best default-threshold model:")
display(cv_summary.head(10))

print("Best tuned threshold:")
display(threshold_summary.head(10))

print("Best ranking results:")
display(ranking_summary.sort_values(["k", "precision_at_k", "ndcg_at_k"], ascending=[True, False, False]).groupby("k").head(5))

print("Interpretation table:")
display(interpretation_table)

### Results

This output gives four tables:

1. default classifier metrics,
2. tuned-threshold metrics,
3. ranking metrics,
4. interpretation table.

### Interpretation

This is what interpretation means in this notebook: not vague commentary, but a decision tied to the metric.

Example: ROC-AUC tells us whether the model separates good and bad jobs. Precision@50 tells us whether the product can show a useful top-50 review queue. F1 tells us whether the binary classifier is strong enough to trust as a guardrail.

## 12. Fit Final Models On Full Dataset

### What this section is

This block fits the selected final score model and classifier on the full dataset after cross-validation has already estimated performance.

### Code

In [ ]:
selected_model_name = str(best_threshold["model"])
selected_threshold = float(best_threshold["threshold"])
selected_spec = next(spec for spec in MODEL_CANDIDATES if spec["name"] == selected_model_name)

final_score_model = make_score_model()
final_score_model.fit(X_base, y_scores)

final_score_predictions = clip_score_predictions(final_score_model.predict(X_base))
final_score_frame = pd.DataFrame(final_score_predictions, columns=SCORE_TARGETS)
X_final_stacked = append_predicted_scores(X_base, final_score_predictions)
model_df["jedi_preference_score"] = compute_jedi_preference_score(final_score_frame)

final_dropped_features, final_corr = low_correlation_engineered_features(
    X_final_stacked,
    y,
    selected_spec["drop_count"],
)

final_scale_pos_weight = float((y == 0).sum() / max((y == 1).sum(), 1))
final_classifier = make_xgb_classifier(
    selected_spec,
    extra_numeric=PRED_SCORE_COLUMNS,
    dropped_engineered=final_dropped_features,
    scale_pos_weight=final_scale_pos_weight,
)
final_classifier.fit(X_final_stacked, y)

final_apply_probability = final_classifier.predict_proba(X_final_stacked)[:, 1]
model_df["apply_probability"] = np.round(final_apply_probability, 4)
model_df["apply_prediction"] = np.where(final_apply_probability >= selected_threshold, "yes", "no")

print("Selected final classifier:", selected_model_name)
print("Selected threshold:", selected_threshold)
print("Dropped engineered features:", len(final_dropped_features))
display(pd.DataFrame({"dropped_feature": final_dropped_features}))

display(
    model_df[["row_id", "company_name", "role_title", "applied", "jedi_preference_score", "apply_probability", "apply_prediction"]]
    .sort_values(["jedi_preference_score", "apply_probability"], ascending=False)
    .head(25)
)

### Results

The output shows the selected model, threshold, dropped engineered features, and a top-25 scored job preview.

### Interpretation

This is the operational form of the model. New scraped jobs should receive the same outputs: JEDI score, apply probability, and recommendation.

## 13. Save Results And Model Artifacts

### What this section is

This block saves the cross-validation tables, scored jobs, model artifacts, and model configuration.

### Code

In [ ]:
cv_summary_path = OUTPUT_DIR / "jedi_final_cv_summary.csv"
threshold_summary_path = OUTPUT_DIR / "jedi_final_threshold_summary.csv"
ranking_summary_path = OUTPUT_DIR / "jedi_final_ranking_summary.csv"
score_metrics_path = OUTPUT_DIR / "jedi_final_score_regression_metrics.csv"
scored_jobs_path = OUTPUT_DIR / "jedi_final_scored_training_jobs.csv"
oof_predictions_path = OUTPUT_DIR / "jedi_final_oof_predictions.csv"
interpretation_path = OUTPUT_DIR / "jedi_final_metric_interpretations.csv"

cv_summary.to_csv(cv_summary_path, index=False)
threshold_summary.to_csv(threshold_summary_path, index=False)
ranking_summary.to_csv(ranking_summary_path, index=False)
score_metrics.to_csv(score_metrics_path, index=False)
oof_predictions.to_csv(oof_predictions_path, index=False)
interpretation_table.to_csv(interpretation_path, index=False)

scored_jobs = model_df[[
    "row_id",
    "company_name",
    "role_title",
    "role_family",
    "seniority",
    "location",
    "applied",
    "jedi_preference_score",
    "apply_probability",
    "apply_prediction",
]].copy()
scored_jobs.to_csv(scored_jobs_path, index=False)

score_model_path = MODEL_DIR / "jedi_ridge_fit_score_model.joblib"
classifier_path = MODEL_DIR / "jedi_xgboost_apply_classifier.joblib"
config_path = MODEL_DIR / "jedi_model_config.json"

dump(final_score_model, score_model_path)
dump(final_classifier, classifier_path)

model_config = {
    "dataset": str(DATA_PATH),
    "score_targets": SCORE_TARGETS,
    "score_max": SCORE_MAX,
    "jedi_score_weights": JEDI_SCORE_WEIGHTS,
    "classifier": selected_model_name,
    "threshold": selected_threshold,
    "dropped_engineered_features": final_dropped_features,
    "predicted_score_columns": PRED_SCORE_COLUMNS,
    "feature_groups": {
        "base_numeric": base_numeric,
        "boolean_features": boolean_features,
        "categorical_features": categorical_features,
        "engineered_features": engineered_features,
    },
}
config_path.write_text(json.dumps(model_config, indent=2), encoding="utf-8")

saved_outputs = pd.DataFrame([
    {"artifact": "cv_summary", "path": str(cv_summary_path)},
    {"artifact": "threshold_summary", "path": str(threshold_summary_path)},
    {"artifact": "ranking_summary", "path": str(ranking_summary_path)},
    {"artifact": "score_regression_metrics", "path": str(score_metrics_path)},
    {"artifact": "metric_interpretations", "path": str(interpretation_path)},
    {"artifact": "scored_jobs", "path": str(scored_jobs_path)},
    {"artifact": "oof_predictions", "path": str(oof_predictions_path)},
    {"artifact": "score_model", "path": str(score_model_path)},
    {"artifact": "classifier", "path": str(classifier_path)},
    {"artifact": "model_config", "path": str(config_path)},
])

display(saved_outputs)

### Results

This output lists every saved file.

### Interpretation

This is the handoff point from notebook to deployable system. The next engineering step is to move this logic into scripts:

- src/features.py,
- src/train.py,
- src/score.py.

The saved model artifacts can then power a scorer API for newly scraped jobs.

## 14. Final Product Decision

### What this section is

This is the final modeling conclusion.

### Code

No code is needed.

### Results

The model produces:

- personal fit-score predictions,
- JEDI preference score,
- apply probability,
- apply recommendation,
- saved CSVs and model artifacts.

### Interpretation

The best product behavior is:

1. score all incoming jobs,
2. sort by JEDI preference score,
3. use apply probability as a guardrail,
4. review the top 25 jobs,
5. store user corrections as new training labels.

Continuous learning should happen weekly or after every 300-500 newly labeled jobs, not after every single daily scrape.